In [1]:
from top2vec import Top2Vec
import json
import time

from nltk.tokenize import word_tokenize
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from itertools import combinations

In [2]:
K_RANGE = list(range(3, 20))
TOP_N = 10

In [3]:
try:
    with open('../dataProcessed/nurseNotes.json', 'r') as file:
        nurse_notes = json.load(file)
    print("File loaded successfully.")
    
except FileNotFoundError:
    print("Error: The file 'data.json' was not found.")

File loaded successfully.


In [4]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [5]:
def top2vec_analysis(texts):
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]
    dictionary = Dictionary(tokenized_texts)

    print(f"Number of texts: {len(texts)}")

    start = time.time()
    top2vec_model = Top2Vec(
        texts,
        embedding_model='all-MiniLM-L6-v2',
        speed="learn"
    )
    cluster_topics = (top2vec_model.get_topics())[0]
    cluster_topics = [topic[:TOP_N] for topic in cluster_topics]
    end = time.time()

    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}")
    print(f"Diversity: {diversity}")
    print(f"Inverse Redundancy: {redundancy}")
    print(f"Time (seconds): {end-start}")

    print("----- Cluster Topics -----")
    for cluster_topic in cluster_topics:
        print(cluster_topic)

    print(f"Number of Topics: {len(cluster_topics)}")

In [6]:
all_texts = []
for key in nurse_notes.keys():
    print(f"-----------{key}-----------")
    top2vec_analysis(nurse_notes[key])
    all_texts.extend(nurse_notes[key])

2026-01-30 16:49:41,216 - top2vec - INFO - Pre-processing documents for training
/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-01-30 16:49:41,249 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model


-----------P1-----------
Number of texts: 600


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-01-30 16:49:43,335 - top2vec - INFO - Creating joint document/word embedding
2026-01-30 16:49:45,333 - top2vec - INFO - Creating lower dimension embedding of documents
2026-01-30 16:49:50,594 - top2vec - INFO - Finding dense areas of documents
2026-01-30 16:49:50,609 - top2vec - INFO - Finding topics
2026-01-30 16:49:52,687 - top2vec - INFO - Pre-processing documents for training
/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-01-30 16:49:52,702 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model


Coherence: 0.4141595379849701
Diversity: 0.6333333333333333
Inverse Redundancy: 0.6000000000000001
Time (seconds): 9.401386022567749
----- Cluster Topics -----
['med' 'resident' 'form' 'care' 'staff' 'medication' 'attend' 'assist'
 'concern' 'chart']
['sleep' 'asleep' 'resident' 'med' 'comfortable' 'care' 'night' 'concern'
 'settle' 'morning']
['asleep' 'sleep' 'toilette' 'resident' 'comfortable' 'morning' 'self'
 'night' 'settle' 'check']
Number of Topics: 3
-----------P10-----------
Number of texts: 625


...

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-01-30 16:51:32,059 - top2vec - INFO - Creating joint document/word embedding
2026-01-30 16:51:33,052 - top2vec - INFO - Creating lower dimension embedding of documents
2026-01-30 16:51:34,159 - top2vec - INFO - Finding dense areas of documents
2026-01-30 16:51:34,174 - top2vec - INFO - Finding topics
2026-01-30 16:51:35,547 - top2vec - INFO - Pre-processing documents for training
/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-01-30 16:51:35,564 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model


Coherence: 0.47632452151994015
Diversity: 0.475
Inverse Redundancy: 0.4833333333333333
Time (seconds): 4.2968878746032715
----- Cluster Topics -----
['resident' 'med' 'wheelchair' 'laxative' 'sit' 'prescribe' 'form'
 'administer' 'medication' 'care']
['bed' 'sleep' 'resident' 'med' 'night' 'comfortable' 'care' 'sit'
 'wheelchair' 'morning']
['bed' 'sleep' 'medication' 'med' 'night' 'settle' 'toilete' 'tts'
 'comfortable' 'resident']
['bed' 'sit' 'sleep' 'medication' 'med' 'wheelchair' 'overnight' 'night'
 'morning' 'prescribe']
Number of Topics: 4
-----------P9-----------
Number of texts: 624


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-01-30 16:51:37,653 - top2vec - INFO - Creating joint document/word embedding
2026-01-30 16:51:38,949 - top2vec - INFO - Creating lower dimension embedding of documents
2026-01-30 16:51:39,427 - top2vec - INFO - Finding dense areas of documents
2026-01-30 16:51:39,436 - top2vec - INFO - Finding topics


Coherence: 0.34334388385088044
Diversity: 0.475
Inverse Redundancy: 0.5
Time (seconds): 3.9000210762023926
----- Cluster Topics -----
['resident' 'med' 'meds' 'compliant' 'medication' 'care' 'administer'
 'concern' 'form' 'maintain']
['adls' 'compliant' 'meds' 'med' 'resident' 'safety' 'settle' 'maintain'
 'administer' 'medication']
['sensor' 'mat' 'safety' 'toilete' 'compliant' 'resident' 'assist' 'need'
 'settle' 'chart']
['resident' 'sensor' 'med' 'compliant' 'safety' 'mat' 'form' 'meds' 'care'
 'concern']
Number of Topics: 4


In [7]:
top2vec_analysis(all_texts)

2026-01-30 16:51:41,153 - top2vec - INFO - Pre-processing documents for training


Number of texts: 12225


/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-01-30 16:51:41,716 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-01-30 16:51:43,556 - top2vec - INFO - Creating joint document/word embedding
2026-01-30 16:51:54,274 - top2vec - INFO - Creating lower dimension embedding of documents
2026-01-30 16:52:04,460 - top2vec - INFO - Finding dense areas of documents
2026-01-30 16:52:04,701 - top2vec - INFO - Finding topics


Coherence: 0.32588235478104055
Diversity: 0.14953271028037382
Inverse Redundancy: 0.7195556339269971
Time (seconds): 23.57634925842285
----- Cluster Topics -----
['resident' 'appointment' 'hospital' 'med' 'form' 'assistance' 'referral'
 'assessment' 'compliant' 'care']
['hospital' 'appointment' 'bed' 'resident' 'sleep' 'med' 'comfort' 'slept'
 'nurse' 'asleep']
['asleep' 'hospital' 'skin' 'sleep' 'resident' 'comfort' 'awake' 'slept'
 'appointment' 'care']
['adls' 'compliant' 'resident' 'meds' 'med' 'hospital' 'safety'
 'appointment' 'ensure' 'settle']
['sleep' 'slept' 'asleep' 'awake' 'bed' 'care' 'relax' 'rest' 'alarm'
 'resident']
['bed' 'medication' 'appointment' 'sleep' 'hospital' 'nurse' 'meds' 'med'
 'slept' 'medicine']
['chart' 'med' 'medication' 'medicine' 'routine' 'meds' 'appointment'
 'assistance' 'hospital' 'antibiotic']
['hospital' 'appointment' 'resident' 'med' 'nurse' 'referral' 'medicine'
 'assistance' 'attend' 'form']
['asleep' 'toileting' 'sleep' 'awake' 'relax' 'slep